# Import Libraries

In [ ]:
import os
import time
import urllib.request
from PIL import Image, UnidentifiedImageError

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

#  Define Image Crawling Function

In [ ]:
def crawl_images(keyword, save_dir, count=500, delay=1.0, scroll_times=30):
    """
    Download images from Google Image Search.
    
    Parameters:
        keyword (str): Search term
        save_dir (str): Directory to save images
        count (int): Number of images to download
        delay (float): Delay between scrolls (seconds)
        scroll_times (int): Number of times to scroll page
    """
    os.makedirs(save_dir, exist_ok=True)

    options = Options()
    # options.add_argument("--headless")  # Uncomment for headless mode
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), options=options
    )
    driver.set_window_size(1920, 1080)

    search_url = f"https://www.google.com/search?q={keyword.replace(' ', '+')}&tbm=isch"
    driver.get(search_url)
    time.sleep(2)

    # Scroll page
    for _ in range(scroll_times):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(delay)
        try:
            driver.find_element(By.CLASS_NAME, "mye4qd").click()
        except:
            pass

    # Collect images
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, "img")))
    images = driver.find_elements(By.CSS_SELECTOR, "img")

    downloaded = 0
    seen_src = set()
    existing_count = len(os.listdir(save_dir))

    for img in images:
        if downloaded >= count:
            break
        try:
            src = img.get_attribute("src") or img.get_attribute("data-src")
            if src and "http" in src and src not in seen_src:
                seen_src.add(src)
                filename = os.path.join(save_dir, f"{keyword.replace(' ', '_')}_{existing_count + downloaded}.jpg")
                urllib.request.urlretrieve(src, filename)
                downloaded += 1
        except Exception:
            continue

    driver.quit()
    print(f"[{keyword}] Downloaded {downloaded} images → {save_dir}")

# Define Activity Map & Run Crawling

In [ ]:

activity_map = {
    "brushing_teeth": "person brushing teeth",
    "drinking": "person drinking water",
    "eating": "person eating food",
    "typing": "person typing on laptop",
    "sleeping": "person sleeping in bed",
    "reading": "person reading book",
    "washing_face": "person washing face",
    "walking": "person walking outside"
}

for label, keyword in activity_map.items():
    folder = f"./images/{label}"
    crawl_images(keyword=keyword, save_dir=folder, count=500, delay=1.2)